# Data Exploration and Quality Checks
**COSC 301 Project — Malaysia State-Level Socioeconomic & Health Outcomes**

This notebook is intentionally focused on understanding the raw data and identifying potential issues before ETL.

Checks included:
1. File-level schema and shape review
2. Missing values, duplicates, and type sanity
3. State and year coverage consistency
4. Key-level join readiness between datasets
5. Potential anomalies to resolve in ETL design

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"

FILES = {
    "income": RAW_DIR / "hh_income_parlimen.csv",
    "poverty": RAW_DIR / "hh_poverty_parlimen.csv",
    "beds": RAW_DIR / "moh_beds.csv",
    "facilities": RAW_DIR / "moh_facilities.csv",
    "population": RAW_DIR / "population_state.csv",
    "worldbank": RAW_DIR / "worldbank_malaysia.csv",
    "income_hist": RAW_DIR / "hh_income_state.csv",
    "poverty_hist": RAW_DIR / "hh_poverty_state.csv",
}

dfs = {name: pd.read_csv(path) for name, path in FILES.items()}

for name, df in dfs.items():
    print(f"{name:12s} rows={len(df):6d} cols={df.shape[1]:2d} file={FILES[name].name}")

income       rows=   666 cols= 5 file=hh_income_parlimen.csv
poverty      rows=   666 cols= 4 file=hh_poverty_parlimen.csv
beds         rows=   149 cols= 8 file=moh_beds.csv
facilities   rows=  3304 cols= 9 file=moh_facilities.csv
population   rows=263679 cols= 6 file=population_state.csv
worldbank    rows=    20 cols= 5 file=worldbank_malaysia.csv
income_hist  rows=   303 cols= 4 file=hh_income_state.csv
poverty_hist rows=   294 cols= 5 file=hh_poverty_state.csv


In [2]:
def profile_table(name, df, n=5):
    print(f"\n=== {name.upper()} ===")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("dtypes:")
    print(df.dtypes.to_string())
    print("sample:")
    display(df.head(n))

for table_name, table_df in dfs.items():
    profile_table(table_name, table_df, n=3)


=== INCOME ===
shape: (666, 5)
columns: ['date', 'state', 'parlimen', 'income_mean', 'income_median']
dtypes:
date               str
state              str
parlimen           str
income_mean      int64
income_median    int64
sample:


,date,state,parlimen,income_mean,income_median
0,2019-01-01,Perlis,P.001 Padang Besar,4747,4075
1,2022-01-01,Perlis,P.001 Padang Besar,4984,4389
2,2024-01-01,Perlis,P.001 Padang Besar,5792,4818



=== POVERTY ===
shape: (666, 4)
columns: ['date', 'state', 'parlimen', 'poverty_absolute']
dtypes:
date                    str
state                   str
parlimen                str
poverty_absolute    float64
sample:


,date,state,parlimen,poverty_absolute
0,2019-01-01,Perlis,P.001 Padang Besar,5.7
1,2022-01-01,Perlis,P.001 Padang Besar,3.7
2,2024-01-01,Perlis,P.001 Padang Besar,3.5



=== BEDS ===
shape: (149, 8)
columns: ['hospital', 'state', 'beds_nonicu', 'util_nonicu', 'beds_icu', 'util_icu', 'vent', 'util_vent']
dtypes:
hospital           str
state              str
beds_nonicu    float64
util_nonicu    float64
beds_icu       float64
util_icu       float64
vent           float64
util_vent      float64
sample:


...

,year,life_expectancy,infant_mortality,health_exp_gdp,hospital_beds
0,2004,74.027,6.6,2.860169,1.89
1,2005,74.370,6.5,2.786927,1.88
2,2006,74.697,6.5,3.108977,1.90



=== INCOME_HIST ===
shape: (303, 4)
columns: ['date', 'state', 'income_mean', 'income_median']
dtypes:
date                 str
state                str
income_mean        int64
income_median    float64
sample:


,date,state,income_mean,income_median
0,1970-01-01,Johor,237,NaN
1,1974-01-01,Johor,382,269.0
2,1976-01-01,Johor,513,370.0



=== POVERTY_HIST ===
shape: (294, 5)
columns: ['date', 'state', 'poverty_absolute', 'poverty_hardcore', 'poverty_relative']
dtypes:
date                    str
state                   str
poverty_absolute    float64
poverty_hardcore    float64
poverty_relative    float64
sample:


,date,state,poverty_absolute,poverty_hardcore,poverty_relative
0,1970-01-01,Johor,45.7,NaN,NaN
1,1976-01-01,Johor,29.0,NaN,NaN
2,1979-01-01,Johor,18.2,NaN,NaN


In [3]:
EXPECTED_STATES = {
    "Johor", "Kedah", "Kelantan", "Melaka", "Negeri Sembilan", "Pahang", "Perak", "Perlis",
    "Pulau Pinang", "Sabah", "Sarawak", "Selangor", "Terengganu",
    "W.P. Kuala Lumpur", "W.P. Labuan", "W.P. Putrajaya",
}

STATE_NORMALIZATION = {
    "W.P Kuala Lumpur": "W.P. Kuala Lumpur",
    "W.P Labuan": "W.P. Labuan",
    "W.P Putrajaya": "W.P. Putrajaya",
    "WP Kuala Lumpur": "W.P. Kuala Lumpur",
    "WP Labuan": "W.P. Labuan",
    "WP Putrajaya": "W.P. Putrajaya",
}


def normalized_states(df, state_col="state"):
    s = df[state_col].astype(str).str.strip().replace(STATE_NORMALIZATION)
    return set(s.dropna().unique())


for name in ["income", "poverty", "beds", "facilities", "population", "income_hist", "poverty_hist"]:
    states = normalized_states(dfs[name])
    missing = sorted(EXPECTED_STATES - states)
    unexpected = sorted(states - EXPECTED_STATES)
    print(f"\n{name.upper()} state coverage")
    print("  count:", len(states))
    print("  missing:", missing)
    print("  unexpected:", unexpected)

print("\n--- Year coverage ---")
for name in ["income", "poverty", "population", "income_hist", "poverty_hist"]:
    if "date" in dfs[name].columns:
        years = sorted(pd.to_datetime(dfs[name]["date"], errors="coerce").dt.year.dropna().unique())
        print(f"{name:12s} years: {years}")

if "year" in dfs["worldbank"].columns:
    wb_years = sorted(pd.to_numeric(dfs["worldbank"]["year"], errors="coerce").dropna().astype(int).unique())
    print("worldbank    years:", wb_years[:3], "...", wb_years[-3:])


INCOME state coverage
  count: 16
  missing: []
  unexpected: []

POVERTY state coverage
  count: 16
  missing: []
  unexpected: []

BEDS state coverage
  count: 16
  missing: []
  unexpected: []

FACILITIES state coverage
  count: 16
  missing: []
  unexpected: []

POPULATION state coverage
  count: 16
  missing: []
  unexpected: []

INCOME_HIST state coverage
  count: 16
  missing: []
  unexpected: []

POVERTY_HIST state coverage
  count: 16
  missing: []
  unexpected: []

--- Year coverage ---
income       years: [np.int32(2019), np.int32(2022), np.int32(2024)]
poverty      years: [np.int32(2019), np.int32(2022), np.int32(2024)]
population   years: [np.int32(1970), np.int32(1971), np.int32(1972), np.int32(1973), np.int32(1974), np.int32(1975), np.int32(1976), np.int32(1977), np.int32(1978), np.int32(1979), np.int32(1980), np.int32(1981), np.int32(1982), np.int32(1983), np.int32(1984), np.int32(1985), np.int32(1986), np.int32(1987), np.int32(1988), np.int32(1989), np.int32(1990), np

In [4]:
def quality_report(name, df, key_cols=None):
    print(f"\n=== {name.upper()} QUALITY ===")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    print("missing columns:")
    print(missing.to_string() if not missing.empty else "  none")

    full_dupes = int(df.duplicated().sum())
    print("duplicate full rows:", full_dupes)

    if key_cols:
        key_dupes = int(df.duplicated(subset=key_cols).sum())
        print(f"duplicate keys ({key_cols}):", key_dupes)

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    if numeric_cols:
        desc = df[numeric_cols].describe().T[["min", "max", "mean"]]
        display(desc.head(20))

quality_report("income", dfs["income"], key_cols=["date", "state", "parlimen"])
quality_report("poverty", dfs["poverty"], key_cols=["date", "state", "parlimen"])
quality_report("beds", dfs["beds"], key_cols=["hospital", "state"])
quality_report("facilities", dfs["facilities"], key_cols=["name", "state"])
quality_report("population", dfs["population"], key_cols=["date", "state", "age", "sex", "ethnicity"])
quality_report("income_hist", dfs["income_hist"], key_cols=["date", "state"])
quality_report("poverty_hist", dfs["poverty_hist"], key_cols=["date", "state"])


=== INCOME QUALITY ===
missing columns:
  none
duplicate full rows: 0
duplicate keys (['date', 'state', 'parlimen']): 0


,min,max,mean
income_mean,3286.0,22319.0,7096.552553
income_median,2346.0,15867.0,5646.430931



=== POVERTY QUALITY ===
missing columns:
  none
duplicate full rows: 0
duplicate keys (['date', 'state', 'parlimen']): 0


,min,max,mean
poverty_absolute,0.0,47.5,8.130781



=== BEDS QUALITY ===
missing columns:
util_icu       93
util_vent      88
util_nonicu    45
beds_nonicu    40
beds_icu       40
vent           40
duplicate full rows: 0
duplicate keys (['hospital', 'state']): 0


...

,min,max,mean
population,0.0,7406.8,36.029511



=== INCOME_HIST QUALITY ===
missing columns:
income_median    11
duplicate full rows: 0
duplicate keys (['date', 'state']): 0


,min,max,mean
income_mean,140.0,13473.0,3342.069307
income_median,108.0,10549.0,2612.565068



=== POVERTY_HIST QUALITY ===
missing columns:
poverty_relative    93
poverty_hardcore    39
poverty_absolute     2
duplicate full rows: 0
duplicate keys (['date', 'state']): 0


,min,max,mean
poverty_absolute,0.0,76.1,12.738356
poverty_hardcore,0.0,15.5,1.827059
poverty_relative,4.9,21.6,14.642289


In [5]:
income = dfs["income"].copy()
poverty = dfs["poverty"].copy()
beds = dfs["beds"].copy()
population = dfs["population"].copy()

for df in [income, poverty, beds, population]:
    if "state" in df.columns:
        df["state"] = df["state"].astype(str).str.strip().replace(STATE_NORMALIZATION)

income_keys = income[["date", "state", "parlimen"]].drop_duplicates()
poverty_keys = poverty[["date", "state", "parlimen"]].drop_duplicates()

missing_in_poverty = income_keys.merge(
    poverty_keys, on=["date", "state", "parlimen"], how="left", indicator=True
).query("_merge == 'left_only'").drop(columns="_merge")

missing_in_income = poverty_keys.merge(
    income_keys, on=["date", "state", "parlimen"], how="left", indicator=True
).query("_merge == 'left_only'").drop(columns="_merge")

print("income keys not found in poverty:", len(missing_in_poverty))
print("poverty keys not found in income:", len(missing_in_income))
display(missing_in_poverty.head(10))
display(missing_in_income.head(10))

pop_overall = population[
    (population["age"] == "overall_age")
    & (population["sex"] == "overall_sex")
    & (population["ethnicity"] == "overall_ethnicity")
]

print("\nPopulation overall rows:", len(pop_overall))
print("Population states:", pop_overall["state"].nunique())
print("Beds states:", beds["state"].nunique())

pop_states = set(pop_overall["state"].dropna().unique())
beds_states = set(beds["state"].dropna().unique())
print("Beds missing population states:", sorted(beds_states - pop_states))

print("\nExploration complete. Use findings above to decide ETL cleaning rules.")

income keys not found in poverty: 0
poverty keys not found in income: 0


,date,state,parlimen


,date,state,parlimen



Population overall rows: 0
Population states: 0
Beds states: 16
Beds missing population states: ['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']

Exploration complete. Use findings above to decide ETL cleaning rules.


## 6. Historical State-Level Data

**Sources:** `hh_income_state.csv`, `hh_poverty_state.csv`

DOSM's officially published state-level HIES aggregates covering survey waves from 1970 to 2022.

Key checks:
- Year alignment between income and poverty series
- Missing value patterns across early survey waves
- State coverage relative to expected 16 states

In [6]:
income_hist = dfs["income_hist"].copy()
poverty_hist = dfs["poverty_hist"].copy()

income_hist["year"] = pd.to_datetime(income_hist["date"]).dt.year
poverty_hist["year"] = pd.to_datetime(poverty_hist["date"]).dt.year

print("income_hist columns :", income_hist.columns.tolist())
print("poverty_hist columns:", poverty_hist.columns.tolist())

# Year alignment: waves present in one but not the other
inc_years = set(income_hist["year"].unique())
pov_years = set(poverty_hist["year"].unique())
print("\nYears in income_hist only :", sorted(inc_years - pov_years))
print("Years in poverty_hist only:", sorted(pov_years - inc_years))
print("Shared years              :", sorted(inc_years & pov_years))

# States per wave: check for uneven state counts across years
print("\nIncome states per year:")
print(income_hist.groupby("year")["state"].nunique().to_string())
print("\nPoverty states per year:")
print(poverty_hist.groupby("year")["state"].nunique().to_string())

income_hist columns : ['date', 'state', 'income_mean', 'income_median', 'year']
poverty_hist columns: ['date', 'state', 'poverty_absolute', 'poverty_hardcore', 'poverty_relative', 'year']

Years in income_hist only : [np.int32(1974)]
Years in poverty_hist only: []
Shared years              : [np.int32(1970), np.int32(1976), np.int32(1979), np.int32(1984), np.int32(1987), np.int32(1989), np.int32(1992), np.int32(1995), np.int32(1997), np.int32(1999), np.int32(2002), np.int32(2004), np.int32(2007), np.int32(2009), np.int32(2012), np.int32(2014), np.int32(2016), np.int32(2019), np.int32(2020), np.int32(2022)]

Income states per year:
year
1970    11
1974    11
1976    14
1979    13
1984    14
1987    14
1989    14
1992    14
1995    14
1997    14
1999    14
2002    14
2004    14
2007    16
2009    16
2012    16
2014    16
2016    16
2019    16
2020    16
2022    16

Poverty states per year:
year
1970    11
1976    13
1979    13
1984    14
1987    14
1989    14
1992    14
1995    14
1997  

In [7]:
# Missing value breakdown by column and by era (pre/post 1990)
print("=== INCOME_HIST missing by column ===")
print(income_hist.isna().sum().to_string())

print("\n=== POVERTY_HIST missing by column ===")
print(poverty_hist.isna().sum().to_string())

print("\n--- poverty_hist: missing poverty_hardcore by year ---")
missing_hc = poverty_hist[poverty_hist["poverty_hardcore"].isna()].groupby("year").size()
print(missing_hc.to_string())

print("\n--- poverty_hist: missing poverty_relative by year ---")
missing_rel = poverty_hist[poverty_hist["poverty_relative"].isna()].groupby("year").size()
print(missing_rel.to_string())

print("\n--- income_hist: missing income_median by year ---")
missing_med = income_hist[income_hist["income_median"].isna()].groupby("year").size()
print(missing_med.to_string())

=== INCOME_HIST missing by column ===
date              0
state             0
income_mean       0
income_median    11
year              0

=== POVERTY_HIST missing by column ===
date                 0
state                0
poverty_absolute     2
poverty_hardcore    39
poverty_relative    93
year                 0

--- poverty_hist: missing poverty_hardcore by year ---
year
1970    11
1976    13
1979    13
1999     1
2002     1

--- poverty_hist: missing poverty_relative by year ---
year
1970    11
1976    13
1979    13
1984    14
1987    14
1989    14
1992    14

--- income_hist: missing income_median by year ---
year
1970    11


In [8]:
# Value ranges across the full historical series
print("=== INCOME_HIST numeric summary ===")
display(income_hist[["income_mean", "income_median"]].describe().T[["min", "max", "mean"]])

print("\n=== POVERTY_HIST numeric summary ===")
display(poverty_hist[["poverty_absolute", "poverty_hardcore", "poverty_relative"]].describe().T[["min", "max", "mean"]])

# Join completeness: how many state-year pairs have both income and poverty
merged_check = income_hist[["state", "year"]].merge(
    poverty_hist[["state", "year"]], on=["state", "year"], how="outer", indicator=True
)
print("\nJoin coverage (income_hist ↔ poverty_hist):")
print(merged_check["_merge"].value_counts().to_string())

print("\nExploration complete.")

=== INCOME_HIST numeric summary ===


,min,max,mean
income_mean,140.0,13473.0,3342.069307
income_median,108.0,10549.0,2612.565068



=== POVERTY_HIST numeric summary ===


,min,max,mean
poverty_absolute,0.0,76.1,12.738356
poverty_hardcore,0.0,15.5,1.827059
poverty_relative,4.9,21.6,14.642289



Join coverage (income_hist ↔ poverty_hist):
_merge
both          291
left_only      12
right_only      3

Exploration complete.
